# Deploy a Flask-based image processing app on AWS SageMaker Asynchronous Inference

### Content
1. Build a Flask Project with a docker file
2. Build and Push Image to Amazon ECR
3. Deploy on SageMaker Asynchronous Inference (using AWS UI)
    - Deploy on SageMaker Asynchronous Inference (using AWS UI)
    - Deploy on SageMaker Asynchronous Inference (using Code)
4. Define auto-scaling for created Endpoint
5. Verify Endpoint
6. Test Endpoint
7. Sagemaker Endpoint Config and Logs

# 1. Build a Flask Project with a docker file
- Check Utils/Sagemaker for sample project files

Documentation:
- Async Inference: https://docs.aws.amazon.com/sagemaker/latest/dg/async-inference-create-endpoint.html
- Docker image commands: https://docs.aws.amazon.com/sagemaker/latest/dg/your-algorithms-inference-code.html

In [7]:
# IAM Roles
# Navigate to AWS IAM > Roles > Create Role > 
# role name: sagemaker-trust
# 1. Permissions tab: AdministratorAccess, AmazonS3FullAccess, AmazonSageMakerFeatureStoreAccess, AmazonSageMakerFullAccess, AmazonEC2ContainerRegistryReadOnly
# 2. Trust Relationship (tab): Add the following json for trust policy

{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "sagemaker.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

{'Version': '2012-10-17',
 'Statement': [{'Effect': 'Allow',
   'Principal': {'Service': 'sagemaker.amazonaws.com'},
   'Action': 'sts:AssumeRole'}]}

# 2. Build and Push Image to Amazon ECR


1. Navigate to AWS Amazon ECR > Private registry > Repositories > Create a private repository > name: flask_demo
2. Build Docker Image locally.
    1. Start docker engine
    2. Start buildx: `docker buildx create --use`
    3. Build docker image:
        1. Build Basic command: `docker build -t flask_demo .`
        2. Build with multi-platform:  `docker build --platform linux/amd64 -t flask_demo --load .`
        3. Update the above command, if it does not work with "docker buildx build"

-----
3. Authenticate AWS
    1. Command: `aws ecr get-login-password --region <region> | docker login --username AWS --password-stdin <aws_account_id>.dkr.ecr.<region>.amazonaws.com`
    2. Example: `aws ecr get-login-password --region us-east-1 | docker login --username AWS --password-stdin 711387092620.dkr.ecr.us-east-1.amazonaws.com/flask_demo`
4. Tag docker image
    1. Command: `docker tag flask_demo:latest [AWS ECR Repository URI]:latest`
    2. Example: `docker tag flask_demo:latest 711387092620.dkr.ecr.us-east-1.amazonaws.com/flask_demo:latest`

----
5. Run image locally
    1.  Command: `docker run -p 8080:8080 flask_demo`
    2.  Run tagged image locally: `docker run -p 8080:8080 711387092620.dkr.ecr.us-east-1.amazonaws.com/flask_demo:latest`
6. Test Image locally with the following commands (To run on Macbook)
    1. Run linux/adm64 docker image on Mac: `docker run --platform=linux/amd64 -it --rm -p 8080:8080 711387092620.dkr.ecr.us-east-1.amazonaws.com/flask_demo:latest`
    2. If Boto3 authentication is required for S3 image download, run the Docker image using the following command:
```sh
docker run --platform=linux/amd64 -it --rm \
    -e AWS_ACCESS_KEY_ID=$(aws configure get aws_access_key_id) \
    -e AWS_SECRET_ACCESS_KEY=$(aws configure get aws_secret_access_key) \
    -e AWS_SESSION_TOKEN=$(aws configure get aws_session_token) \
    -p 8080:8080 711387092620.dkr.ecr.us-east-1.amazonaws.com/flask_demo:latest serve


Note: Use serve in the end to mimic the same behavior as of AWS sagemaker command.


---
7. Push docker image
   1. `docker push [AWS ECR Repository URI]:latest`
    2. Example: `docker push 711387092620.dkr.ecr.us-east-1.amazonaws.com/flask_demo:latest`
8. Verify docker image in AWS console.
   1. Via UI, Open AWS ECR, there should be a new entry: https://us-east-1.console.aws.amazon.com/ecr/private-registry/repositories?region=us-east-1
    2. Use the following commands to check for manifest type (it should be: application/vnd.docker.distribution.manifest.v2+json)
       - Example: `docker manifest inspect 711387092620.dkr.ecr.us-east-1.amazonaws.com/flask_demo:latest`

# 3. Deploy on SageMaker Asynchronous Inference (using AWS UI)

#### Create a Model in SageMaker
1. Navigate to AWS SageMaker AI -> Inference (left side tab) -> Models -> Create Model
2. Model Configuration
    1. Model Name: flask-demo-model
    2. IAM Role: Sagemaker-trust    (created above)
    3. Location of inference code image: ECR image URL
        - Example: 711387092620.dkr.ecr.us-east-1.amazonaws.com/flask_demo
    4. Config
        1. Model Compression Type: UnCompressed 
    6. Location of model artifacts: dummy S3 bucket path

#### Create a Endpoint in SageMaker
1. Navigate to AWS SageMaker -> Endpoint Configurations -> Create Configuration
2. Endpoint Configuration
    1. Endpoint configuration name: flask-demo-endpoint
    2. Type of endpoint: Provisioned
    3. Async Invocation Config: Select it and provide following config
    4. Async Invocation Config > Max concurrent invocations per instance: 1
    5. Async Invocation Config > S3 output path: s3://flask-demo-bucket-99989/response/
    6. Variants > Create Production variant > Add: flask-demo-model    # Select the same model created above
    7. To select CPU/GPU
        1. Variants > Production, then scroll and click on edit.
    8. Create endpoint configuration 

#### Deploy an Endpoint
1. Navigate to AWS SageMaker -> Endpoint -> Create Endpoint
2. Configuration
    1. Endpoint name: flask-demo-endpoint-prod
    2. Attach endpoint configuration: Select the existing configured endpoint.
    3. Create
4. Check status of created endpoint.
    1. Type should be: Async 
 

    


# 3. Deploy on SageMaker Asynchronous Inference (using Code)

### Create Model, Endpoint Configuration, Deploy Endpoint

In [88]:
# Config

# Model Config
model_name = "flask-demo-model"
ecr_repo_name = "flask_demo"
ecr_image = "711387092620.dkr.ecr.us-east-1.amazonaws.com/flask_demo:latest"
iam_role = "arn:aws:iam::711387092620:role/sagemaker-trust"  # Copy ARN of new role created above

# Endpoint Config
endpoint_config_name = "flask-demo-config-endpoint"
endpoint_name = "flask-demo-endpoint-prod"
output_s3_bucket = "s3://flask-demo-bucket-99989/response/"

In [89]:
import boto3
sagemaker = boto3.client("sagemaker", region_name="us-east-1")
sagemaker

In [1]:
# Delete and Create ECR repo, and Delete existing Model, Endpoint Configuration, and Endpoint

try:
    # Delete and create ECR repo
    ecr_client = boto3.client("ecr", region_name="us-east-1")  # Change region if needed
    ecr_client.delete_repository(repositoryName=ecr_repo_name, force=True)
    #response = ecr_client.create_repository(repositoryName=ecr_repo_name)
    #print("ecr_client", response )
except Exception as e:
    print("ECR repository deletion error: ", e)

try:
    sagemaker.delete_endpoint(EndpointName=endpoint_name)
except Exception as e:
    print("Endpoint deletion error: ", e)

try:
    sagemaker.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
except Exception as e:
    print("EndpointConfiguration deletion error:", e)

try:
    sagemaker.delete_model(ModelName=model_name)
except Exception as e:
    print("Model deletion error:", e)


ECR repository deletion error:  name 'boto3' is not defined
Endpoint deletion error:  name 'sagemaker' is not defined
EndpointConfiguration deletion error: name 'sagemaker' is not defined
Model deletion error: name 'sagemaker' is not defined


In [101]:
# Step 1: Create SageMaker model
created_model = sagemaker.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image": ecr_image,
        #"ModelDataUrl": "",
        'Mode': 'SingleModel',
        'Environment': {
            'SAGEMAKER_PROGRAM': 'app.py'
        }
    },
    ExecutionRoleArn=iam_role,
)

print("✅ Step 1: Sagemaker Model Created", created_model)


✅ Step 1: Sagemaker Model Created {'ModelArn': 'arn:aws:sagemaker:us-east-1:711387092620:model/flask-demo-model', 'ResponseMetadata': {'RequestId': 'e68e3982-5634-470c-8921-b7fe87d117ef', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'e68e3982-5634-470c-8921-b7fe87d117ef', 'content-type': 'application/x-amz-json-1.1', 'content-length': '78', 'date': 'Sat, 08 Feb 2025 10:17:29 GMT'}, 'RetryAttempts': 0}}


In [102]:
# Step 2: Create Endpoint Config without AutoScalingConfig
endpoint_config_response = sagemaker.create_endpoint_config(
    EndpointConfigName = endpoint_config_name,
    AsyncInferenceConfig={
        "OutputConfig": {
            "S3OutputPath": output_s3_bucket
        },
        "ClientConfig": {
            "MaxConcurrentInvocationsPerInstance": 1
        }
    },
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InstanceType": 'ml.m4.xlarge',
            "InitialInstanceCount": 1,  # Default is 1, we will scale it to 0 later
            "ContainerStartupHealthCheckTimeoutInSeconds": 1200
        }
    ]
)

print("✅ Step 2: Endpoint Configuration Created", endpoint_config_response)

✅ Step 2: Endpoint Configuration Created {'EndpointConfigArn': 'arn:aws:sagemaker:us-east-1:711387092620:endpoint-config/flask-demo-config-endpoint', 'ResponseMetadata': {'RequestId': 'aa7583ef-d92f-461c-884f-7ae82e94df63', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'aa7583ef-d92f-461c-884f-7ae82e94df63', 'content-type': 'application/x-amz-json-1.1', 'content-length': '107', 'date': 'Sat, 08 Feb 2025 10:17:29 GMT'}, 'RetryAttempts': 0}}


In [103]:
# Step 3: Deploy the Endpoint
endpoint_response = sagemaker.create_endpoint(
    EndpointName = endpoint_name,
    EndpointConfigName = endpoint_config_name
)

print("✅ Step 3: Endpoint Deployed:", endpoint_response)

✅ Step 3: Endpoint Deployed: {'EndpointArn': 'arn:aws:sagemaker:us-east-1:711387092620:endpoint/flask-demo-endpoint-prod', 'ResponseMetadata': {'RequestId': 'd0cb3269-b247-4d2e-8fba-111f7278b4c3', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'd0cb3269-b247-4d2e-8fba-111f7278b4c3', 'content-type': 'application/x-amz-json-1.1', 'content-length': '92', 'date': 'Sat, 08 Feb 2025 10:17:30 GMT'}, 'RetryAttempts': 0}}


# 4. Define auto-scaling for created Endpoint

In [107]:
autoscaling = boto3.client("application-autoscaling")
endpoint_resource_id = f"endpoint/{endpoint_name}/variant/AllTraffic"

# Step 4: Register auto-scaling
auto_scaling_response1 = autoscaling.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=endpoint_resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=0,   # Allow scaling down to 0 to save cost. The minimum number of instances that the endpoint should maintain.
    MaxCapacity=1   # Scale up to 5 instance during peak. The maximum number of instances that the endpoint can scale up to.
)

# Step 5: Define Scaling Policy (Target-based Scaling)
auto_scaling_response2 = autoscaling.put_scaling_policy(
    PolicyName="TargetTrackingScaling",
    ServiceNamespace="sagemaker",
    ResourceId=endpoint_resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 2,  # Average 2 requests per instance. # The target value for the scaling metric. This is typically the average number of requests per instance. SageMaker will try to maintain an average of 2 requests per instance.
        "PredefinedMetricSpecification": {
            "PredefinedMetricType": "SageMakerVariantInvocationsPerInstance"
        },
        "ScaleInCooldown": 60,  # Scale down after 1 minutes of inactivity means SageMaker will wait 1 minute after scaling down before it can scale down again.
        "ScaleOutCooldown": 60   # Scale up if needed, but wait 1 minute, means SageMaker will wait 1 minute after scaling up before it can scale up again.
    }
)

print("✅ Auto-scaling enabled for", endpoint_name)
print("✅ Endpoint ResourceId", endpoint_resource_id)
print("✅ Step 4: Response 1", auto_scaling_response1)
print("✅ Step 5: Response 2", auto_scaling_response2)

✅ Auto-scaling enabled for flask-demo-endpoint-prod
✅ Endpoint ResourceId endpoint/flask-demo-endpoint-prod/variant/AllTraffic
✅ Step 4: Response 1 {'ScalableTargetARN': 'arn:aws:application-autoscaling:us-east-1:711387092620:scalable-target/056m66999066b6314c308a18c0c26712a4b4', 'ResponseMetadata': {'RequestId': 'ce1af459-7b17-4413-8310-b16a8ed4c03d', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'ce1af459-7b17-4413-8310-b16a8ed4c03d', 'content-type': 'application/x-amz-json-1.1', 'content-length': '131', 'date': 'Sat, 08 Feb 2025 10:22:58 GMT'}, 'RetryAttempts': 0}}
✅ Step 5: Response 2 {'PolicyARN': 'arn:aws:autoscaling:us-east-1:711387092620:scalingPolicy:66999066-b631-4c30-8a18-c0c26712a4b4:resource/sagemaker/endpoint/flask-demo-endpoint-prod/variant/AllTraffic:policyName/TargetTrackingScaling', 'Alarms': [{'AlarmName': 'TargetTracking-endpoint/flask-demo-endpoint-prod/variant/AllTraffic-AlarmHigh-4f00db0f-ec70-4486-8e4e-78572dd53b20', 'AlarmARN': 'arn:aws:cloudwatch:

# 5. Verify Endpoint

In [108]:
# List Endpoints
response = sagemaker.list_endpoints(
    NameContains = endpoint_name
)

print(response)

{'Endpoints': [{'EndpointName': 'flask-demo-endpoint-prod', 'EndpointArn': 'arn:aws:sagemaker:us-east-1:711387092620:endpoint/flask-demo-endpoint-prod', 'CreationTime': datetime.datetime(2025, 2, 8, 15, 47, 30, 472000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(2025, 2, 8, 15, 50, 56, 756000, tzinfo=tzlocal()), 'EndpointStatus': 'InService'}], 'ResponseMetadata': {'RequestId': '6be9a053-592e-49ed-9645-6812fb0bf6c1', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '6be9a053-592e-49ed-9645-6812fb0bf6c1', 'content-type': 'application/x-amz-json-1.1', 'content-length': '247', 'date': 'Sat, 08 Feb 2025 10:23:03 GMT'}, 'RetryAttempts': 0}}


In [109]:
# Endpoint Status
response = sagemaker.describe_endpoint(EndpointName = endpoint_name)
print("Endpoint Status:", response["EndpointStatus"])
print(response)

Endpoint Status: InService
{'EndpointName': 'flask-demo-endpoint-prod', 'EndpointArn': 'arn:aws:sagemaker:us-east-1:711387092620:endpoint/flask-demo-endpoint-prod', 'EndpointConfigName': 'flask-demo-config-endpoint', 'ProductionVariants': [{'VariantName': 'AllTraffic', 'DeployedImages': [{'SpecifiedImage': '711387092620.dkr.ecr.us-east-1.amazonaws.com/flask_demo:latest', 'ResolvedImage': '711387092620.dkr.ecr.us-east-1.amazonaws.com/flask_demo@sha256:cc07fddf7e9d72aa2c8ce34036593d35a46c044b321ccbb03d5b8e373476f2de', 'ResolutionTime': datetime.datetime(2025, 2, 8, 15, 47, 31, 169000, tzinfo=tzlocal())}], 'CurrentWeight': 1.0, 'DesiredWeight': 1.0, 'CurrentInstanceCount': 1, 'DesiredInstanceCount': 1}], 'EndpointStatus': 'InService', 'CreationTime': datetime.datetime(2025, 2, 8, 15, 47, 30, 472000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(2025, 2, 8, 15, 50, 56, 756000, tzinfo=tzlocal()), 'AsyncInferenceConfig': {'ClientConfig': {'MaxConcurrentInvocationsPerInstance': 1},

In [110]:
# Verify auto-scaling
response = autoscaling.describe_scalable_targets(
    ServiceNamespace="sagemaker",
    ResourceIds=[endpoint_resource_id]
)
print(response)

{'ScalableTargets': [{'ServiceNamespace': 'sagemaker', 'ResourceId': 'endpoint/flask-demo-endpoint-prod/variant/AllTraffic', 'ScalableDimension': 'sagemaker:variant:DesiredInstanceCount', 'MinCapacity': 0, 'MaxCapacity': 1, 'RoleARN': 'arn:aws:iam::711387092620:role/aws-service-role/sagemaker.application-autoscaling.amazonaws.com/AWSServiceRoleForApplicationAutoScaling_SageMakerEndpoint', 'CreationTime': datetime.datetime(2025, 2, 8, 15, 52, 58, 780000, tzinfo=tzlocal()), 'SuspendedState': {'DynamicScalingInSuspended': False, 'DynamicScalingOutSuspended': False, 'ScheduledScalingSuspended': False}, 'ScalableTargetARN': 'arn:aws:application-autoscaling:us-east-1:711387092620:scalable-target/056m66999066b6314c308a18c0c26712a4b4'}], 'ResponseMetadata': {'RequestId': 'd69c090f-50d0-4ae6-826d-7eb079b1d524', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'd69c090f-50d0-4ae6-826d-7eb079b1d524', 'content-type': 'application/x-amz-json-1.1', 'content-length': '664', 'date': 'Sat, 08

# 6. Test Endpoint

In [69]:
# Not working
response = requests.post(
          sagemaker_endpoint,
          auth=AWS4Auth(
              access_id,
              access_key,
              region,
              "sagemaker",
              session_token,
          ),
          json=some_json_request,
          headers={
              "X-Amzn-SageMaker-Custom-Attributes": 'custom header value'
          },
      )

Server Response: {'message': 'Missing Authentication Token'}


In [112]:
# Working
# Upload Request json to S3 and then call the endpoint

import json
s3 = boto3.client("s3")
bucket_name = "flask-demo-bucket-99989"
json_data = {
    "input_image": "input.png",
    "output_image": "output3.png"
}

json_file_path = "input.json"
with open(json_file_path, "w") as f:
    json.dump(json_data, f)

# Upload JSON file to S3
s3.upload_file(json_file_path, bucket_name, "input.json")

s3_input_uri = f"s3://{bucket_name}/input.json"
print("Request Json uploaded to S3:", s3_input_uri)





import boto3

sagemaker_runtime = boto3.client("sagemaker-runtime")

response = sagemaker_runtime.invoke_endpoint_async(
    EndpointName=endpoint_name,
    CustomAttributes='applicationCustom',
    InputLocation=s3_input_uri  # JSON input stored in S3
)

print(response["InferenceId"])

Request Json uploaded to S3: s3://flask-demo-bucket-99989/input.json
b0d2ef9e-ced9-4523-8026-91a538b8aca9


In [113]:
# Check logs

import boto3

logs_client = boto3.client("logs")

log_group = f"/aws/sagemaker/Endpoints/{endpoint_name}"

response = logs_client.describe_log_streams(
    logGroupName=log_group,
    orderBy="LastEventTime",
    descending=True
)

if response["logStreams"]:
    log_stream = response["logStreams"][0]["logStreamName"]  # Get latest log

    logs = logs_client.get_log_events(
        logGroupName=log_group,
        logStreamName=log_stream,
        limit=10
    )

    for event in logs["events"]:
        print(event["message"])
else:
    print("No logs found for this endpoint.")

# Filter Logs in AWS Logs UI
# Add -"/ping" in the seachbox and press enter.

2025-02-08T10:23:19.837:[sagemaker logs] [b1615575-f09e-441d-9aaf-960120ca6456] Inference request succeeded. ModelLatency: 457922 us, RequestDownloadLatency: 375596 us, ResponseUploadLatency: 379686 us, TimeInBacklog: 12 ms, TotalProcessingTime: 1437 ms
2025-02-08T10:25:09.491:[sagemaker logs] [b0d2ef9e-ced9-4523-8026-91a538b8aca9] Inference request succeeded. ModelLatency: 571460 us, RequestDownloadLatency: 41430 us, ResponseUploadLatency: 72682 us, TimeInBacklog: 10 ms, TotalProcessingTime: 703 ms


In [89]:
# Download processed image

output_file = "output.png"

s3 = boto3.client("s3")
s3.download_file(bucket_name, output_file, output_file)


# 7. Sagemaker Endpoint Config and Logs

1. Check auto-scaling policy, if enabled and maximum number of instances.
    - Command: `aws application-autoscaling describe-scalable-targets --service-namespace sagemaker`
2. Check current instance count with the specific endpoint
    1. Command: `aws sagemaker describe-endpoint --endpoint-name flask-image-processor-endpoint --query "ProductionVariants"`
    2. Example: `aws sagemaker describe-endpoint --endpoint-name flask-demo-endpoint-prod --query "ProductionVariants"`

#### Check Logs
1. Check logs, Command: `aws sagemaker describe-endpoint --endpoint-name flask-demo-endpoint-prod`
2. Naviage to Amazon SageMaker AI > Endpoints > flask-demo-endpoint-prod
    1. For deployed endpoint, check "Model container logs"
